# H&M Customer Profiling
## Developing Repeat vs. Recent Occasional Customers

This notebook compares the historical profiles of two RFM segments. It screens candidate variables for the next logistic-regression notebook.

All profiling variables are measured during the historical observation window. Future 120-day repurchase behaviour is not used to create the profiles or screen the initial candidate variables; it is joined later as the predictive outcome.

## 1. Load the data

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

In [2]:
data_path = Path("/Volumes/USB/H&M Data/customer_features.csv")
df = pd.read_csv(data_path)

In [3]:
df = df.rename(columns={"entropy": "product_group_entropy", "fn": "fn_flag"})
df["age_missing"] = df["age"].isna().astype(int)

In [4]:
print("Rows:", len(df))
print("Unique customers:", df["customer_id"].nunique())
display(df.head())

Rows: 608482
Unique customers: 608482


,customer_id,segment,product_group_entropy,fn_flag,active,age,club_member_active,regular_fashion_news,shopping_days,item_count,...,distinct_articles,distinct_garment_groups,distinct_product_groups,distinct_index_groups,both_channels,channel_2_only,channel_1_only,channel_2_item_share,unique_article_share,age_missing
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,1,0.895333,0,0,49.0,1,0,6,14,...,13,9,4,3,1,0,0,0.642857,0.928571,0
1,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,1,1.010100,0,0,24.0,1,0,3,13,...,10,6,3,3,0,1,0,1.000000,0.769231,0
2,00007e8d4e54114b5b2a9b51586325a8d0fa74ea23ef77...,2,0.693147,0,0,20.0,1,0,1,2,...,2,2,2,1,0,0,1,0.000000,1.000000,0
3,0000b2f1829e23b24feec422ef13df3ccedaedc85368e6...,1,1.430562,1,1,54.0,1,1,5,20,...,15,8,5,3,1,0,0,0.500000,0.750000,0
4,0000c97821eb48d0e590fd309133f0a6c08f7750f64ccc...,1,1.090599,0,0,49.0,1,0,4,13,...,13,4,4,2,1,0,0,0.923077,1.000000,0


## 2. Validate the customer-level dataset

In [5]:
assert df["customer_id"].is_unique
assert set(df["segment"].unique()) == {1, 2}

In [6]:
channel_total = df["channel_1_only"] + df["channel_2_only"] + df["both_channels"]
assert channel_total.eq(1).all()

In [7]:
display(df["segment"].value_counts().sort_index().to_frame("customers"))
display(df.isna().sum().loc[lambda x: x > 0].to_frame("missing_values"))

,customers
segment,
1,279695
2,328787


,missing_values
age,3815


## 3. Define feature groups

`shopping_days` is the historical RFM Frequency measure. `spend_index` is the historical RFM Monetary measure. They remain in the descriptive profile, but the next notebook will use the canonical `log_frequency` and `log_monetary` fields as control variables.

In [8]:
feature_groups = {
    "Customer Profile": ["age", "age_missing", "fn_flag", "active", "club_member_active", "regular_fashion_news"],
    "Purchase Intensity": ["shopping_days", "item_count", "spend_index", "items_per_shopping_day"],
    "Product Exploration": ["distinct_articles", "distinct_garment_groups", "distinct_product_groups", "distinct_index_groups", "product_group_entropy"],
    "Channel Behaviour": ["channel_1_only", "channel_2_only", "both_channels", "channel_2_item_share"],
    "Article Variety": ["unique_article_share"]
}

## 4. Descriptive profiling

A positive SMD means the variable is higher among Developing Repeat customers. Absolute SMD describes the size of the difference.

In [9]:
def calculate_smd(group_1, group_2):
    variance_1 = group_1.var(ddof=1)
    variance_2 = group_2.var(ddof=1)
    pooled_sd = np.sqrt((variance_1 + variance_2) / 2)
    return (group_1.mean() - group_2.mean()) / pooled_sd

In [10]:
def classify_smd(value):
    if value < 0.10: return "Minimal"
    if value < 0.20: return "Small"
    if value < 0.50: return "Moderate"
    return "Large"

In [11]:
developing = df[df["segment"] == 1]
recent = df[df["segment"] == 2]

In [12]:
profile_rows = []

In [13]:
for feature_group, features in feature_groups.items():
    for feature in features:
        developing_mean = developing[feature].mean()
        recent_mean = recent[feature].mean()
        smd = calculate_smd(developing[feature].dropna(), recent[feature].dropna())
        profile_rows.append([feature_group, feature, developing_mean, recent_mean, developing_mean - recent_mean, smd, abs(smd), classify_smd(abs(smd))])

In [14]:
profile_columns = ["feature_group", "feature", "developing_repeat_mean", "recent_occasional_mean", "mean_difference", "smd", "absolute_smd", "difference_magnitude"]
profiling_table = pd.DataFrame(profile_rows, columns=profile_columns)

In [15]:
profiling_display = profiling_table.set_index(["feature_group", "feature"])
display(profiling_display.round(3))

developing_repeat_mean  \
feature_group       feature                                           
Customer Profile    age                                      35.736   
                    age_missing                               0.005   
                    fn_flag                                   0.425   
                    active                                    0.419   
                    club_member_active                        0.987   
                    regular_fashion_news                      0.426   
Purchase Intensity  shopping_days                             6.586   
                    item_count                               20.677   
                    spend_index                               0.539   
                    items_per_shopping_day                    3.540   
Product Exploration distinct_articles                        18.276   
                    distinct_garment_groups                   7.337   
                    distinct_product_groups                   4.581   
                    distinct_index_groups                     2.613   
                    product_group_entropy                     1.216   
Channel Behaviour   channel_1_only                            0.121   
                    channel_2_only                            0.269   
                    both_channels                             0.610   
                    channel_2_item_share                      0.611   
Article Variety     unique_article_share                      0.894   

                                             recent_occasional_mean  \
feature_group       feature                                           
Customer Profile    age                                      35.256   
                    age_missing                               0.008   
                    fn_flag                                   0.320   
                    active                                    0.313   
                    club_member_active                        0.943   
                    regular_fashion_news                      0.320   
Purchase Intensity  shopping_days                             1.807   
                    item_count                                5.450   
                    spend_index                               0.143   
                    items_per_shopping_day                    3.226   
Product Exploration distinct_articles                         4.877   
                    distinct_garment_groups                   2.802   
                    distinct_product_groups                   2.231   
                    distinct_index_groups                     1.638   
                    product_group_entropy                     0.587   
Channel Behaviour   channel_1_only                            0.201   
                    channel_2_only                            0.653   
                    both_channels                             0.146   
                    channel_2_item_share                      0.731   
Article Variety     unique_article_share                      0.923   

                                             mean_difference    smd  \
feature_group       feature                                           
Customer Profile    age                                0.481  0.034   
                    age_missing                       -0.003 -0.041   
                    fn_flag                            0.105  0.219   
                    active                             0.105  0.220   
                    club_member_active                 0.044  0.243   
                    regular_fashion_news               0.106  0.221   
Purchase Intensity  shopping_days                      4.779  2.262   
                    item_count                        15.227  1.990   
                    spend_index                        0.396  1.971   
                    items_per_shopping_day             0.314  0.121   
Product Exploration distinct_articles                 13.399  1.998

### Descriptive interpretation

- Large purchase-intensity differences are expected because RFM behaviour was used to create the original customer segments.
- Developing Repeat customers are more likely to use both channels. Recent Occasional customers are more concentrated in Channel 2.
- Developing Repeat customers explore a broader range of products, although raw distinct counts also increase with purchase volume.
- Several customer-engagement fields may describe almost the same behaviour.

## 5. Redundancy analysis

In [16]:
redundancy_groups = {
    "Customer Engagement": ["fn_flag", "active", "regular_fashion_news", "club_member_active"],
    "Purchase Intensity": ["shopping_days", "item_count", "spend_index", "items_per_shopping_day", "distinct_articles"],
    "Product Exploration": ["distinct_garment_groups", "distinct_product_groups", "distinct_index_groups", "product_group_entropy"],
    "Channel Behaviour": ["channel_1_only", "channel_2_only", "both_channels", "channel_2_item_share"]
}

In [17]:
correlation_rows = []

In [18]:
for feature_group, features in redundancy_groups.items():
    correlation_matrix = df[features].corr()
    for i in range(len(features)):
        for j in range(i + 1, len(features)):
            feature_1 = features[i]
            feature_2 = features[j]
            correlation = correlation_matrix.loc[feature_1, feature_2]
            correlation_rows.append([feature_group, feature_1, feature_2, correlation, abs(correlation)])

In [19]:
correlation_columns = ["feature_group", "feature_1", "feature_2", "correlation", "absolute_correlation"]
redundancy_table = pd.DataFrame(correlation_rows, columns=correlation_columns)

In [20]:
group_order = list(redundancy_groups.keys())
redundancy_table["feature_group"] = pd.Categorical(redundancy_table["feature_group"], categories=group_order, ordered=True)

In [21]:
redundancy_table = redundancy_table.sort_values(["feature_group", "absolute_correlation"], ascending=[True, False])
redundancy_table["pair_number"] = redundancy_table.groupby("feature_group", observed=True).cumcount() + 1

In [22]:
redundancy_display = redundancy_table.set_index(["feature_group", "pair_number"])
redundancy_display = redundancy_display[["feature_1", "feature_2", "correlation", "absolute_correlation"]]

In [23]:
redundancy_styled = redundancy_display.style.hide(axis="index", level=1)
redundancy_styled = redundancy_styled.format({"correlation": "{:.3f}", "absolute_correlation": "{:.3f}"})

In [24]:
redundancy_styles = [{"selector": "th.row_heading.level0", "props": [("vertical-align", "top"), ("text-align", "left"), ("font-weight", "bold")]}]
redundancy_styled = redundancy_styled.set_table_styles(redundancy_styles)
display(redundancy_styled)

## 6. Variable-screening decisions

Decisions use descriptive differences, SMD, redundancy, overlap with RFM controls, purchase-volume exposure, and business interpretation. 

In [25]:
decisions = [
 ["age", "Customer profile", "Exclude initially", "Minimal difference between the two customer groups and a very low SMD."],
 ["age_missing", "Data-quality indicator", "Exclude initially", "Missing-age rates are very similar between the groups and the SMD is minimal."],
 ["fn_flag", "Engagement", "Exclude initially", "Near-duplicate engagement field."],
 ["active", "Engagement", "Exclude initially", "Near-duplicate engagement field."],
 ["club_member_active", "Membership candidate", "Carry forward", "Distinct membership relationship."],
 ["regular_fashion_news", "CRM engagement candidate", "Carry forward", "Most interpretable representative of the engagement flags."],
 ["shopping_days", "RFM Frequency", "Replace with RFM control", "Use canonical log_frequency."],
 ["item_count", "Purchase intensity", "Exclude initially", "Strong overlap with RFM intensity and distinct articles."],
 ["spend_index", "RFM Monetary", "Replace with RFM control", "Use canonical log_monetary."],
 ["items_per_shopping_day", "Basket-depth candidate", "Carry forward provisionally", "Represents basket depth rather than frequency."],
 ["distinct_articles", "Product exploration", "Exclude initially", "Highly dependent on purchase volume."],
 ["distinct_garment_groups", "Product exploration", "Exclude initially", "Use entropy as the representative diversity measure."],
 ["distinct_product_groups", "Product exploration", "Exclude initially", "Strong overlap with entropy and purchase volume."],
 ["distinct_index_groups", "Product exploration", "Exclude initially", "Overlaps with other breadth measures."],
 ["product_group_entropy", "Product-diversity candidate", "Carry forward", "Measures the breadth and balance of product exploration."],
 ["channel_1_only", "Channel behaviour", "Descriptive only", "Retain the descriptive channel insight without duplicating predictors."],
 ["channel_2_only", "Channel behaviour", "Descriptive only", "Shows Channel 2 concentration but overlaps with channel breadth."],
 ["both_channels", "Cross-channel candidate", "Carry forward", "Most interpretable cross-channel measure."],
 ["channel_2_item_share","Channel-preference candidate","Carry forward provisionally","Captures Channel 2 purchasing concentration and will be tested for incremental value alongside both_channels."],
 ["unique_article_share", "Article variety", "Exclude initially", "Weak business interpretation and mechanically related to item count."]
]

In [26]:
decision_columns = ["feature", "model_role", "decision", "reason"]
decision_table = pd.DataFrame(decisions, columns=decision_columns)

In [27]:
selection_columns = ["feature_group", "feature", "smd", "absolute_smd"]
feature_selection_table = profiling_table[selection_columns].merge(decision_table, on="feature", validate="one_to_one")

In [28]:
ordered_pairs = [(group, feature) for group, features in feature_groups.items() for feature in features]
ordered_index = pd.MultiIndex.from_tuples(ordered_pairs, names=["feature_group", "feature"])

In [29]:
selection_display = feature_selection_table.set_index(["feature_group", "feature"])
selection_display = selection_display.reindex(ordered_index)
selection_display = selection_display[["smd", "absolute_smd", "model_role", "decision", "reason"]]

In [30]:
selection_styled = selection_display.style.format({"smd": "{:.3f}", "absolute_smd": "{:.3f}"})

In [31]:
selection_styles = [{"selector": "th.row_heading.level0", "props": [("vertical-align", "top"), ("text-align", "left"), ("font-weight", "bold")]}]
selection_styled = selection_styled.set_table_styles(selection_styles)
display(selection_styled)

## 7. Variables carried forward

This is the final output of the profiling notebook. These are initial modelling inputs, not variables retained by a completed logistic regression.

In [32]:
model_inputs = [
 ["RFM Controls", "recency", "Main specification"],
 ["RFM Controls", "log_frequency", "Main specification"],
 ["RFM Controls", "log_monetary", "Main specification"],
 ["Candidate Signals", "regular_fashion_news", "Main specification"],
 ["Candidate Signals", "club_member_active", "Main specification"],
 ["Candidate Signals", "product_group_entropy", "Main specification"],
 ["Candidate Signals", "both_channels", "Main specification"],
 ["Additional Candidate","items_per_shopping_day","Test incremental value"],
["Additional Candidate","channel_2_item_share","Test incremental value alongside both_channels"]
]

In [33]:
model_input_columns = ["variable_group", "variable", "model_use"]
model_input_table = pd.DataFrame(model_inputs, columns=model_input_columns)
model_input_table["row_number"] = model_input_table.groupby("variable_group", sort=False).cumcount() + 1

In [34]:
model_input_display = model_input_table.set_index(["variable_group", "row_number"])
model_input_display = model_input_display[["variable", "model_use"]]

In [35]:
model_input_styled = model_input_display.style.hide(axis="index", level=1)
model_input_styles = [{"selector": "th.row_heading.level0", "props": [("vertical-align", "top"), ("text-align", "left"), ("font-weight", "bold")]}]
model_input_styled = model_input_styled.set_table_styles(model_input_styles)
display(model_input_styled)

## 8. Create the Logistic-Regression Dataset

In [36]:
rfm_path = Path("/Volumes/USB/H&M Data/hm_customer_segments.csv")
rfm_columns = ["customer_id","recency","log_frequency","log_monetary","repurchase","cluster"]
rfm_data = pd.read_csv(rfm_path,usecols=rfm_columns)

In [37]:
model_data = df.merge(rfm_data,on="customer_id",how="inner",validate="one_to_one")

In [38]:
assert len(model_data) == len(df)
assert model_data["segment"].eq(model_data["cluster"]).all()

In [39]:
model_data = model_data.drop(columns="cluster")

In [40]:
model_data = model_data[model_data["segment"] == 2].copy()

In [41]:
model_columns = [
    "customer_id",
    "segment",
    "repurchase",
    "recency",
    "log_frequency",
    "log_monetary",
    "regular_fashion_news",
    "club_member_active",
    "product_group_entropy",
    "both_channels",
    "items_per_shopping_day",
    "channel_2_item_share"
]

model_data = model_data[model_columns]

In [42]:
print("Rows:", len(model_data))
print("Unique customers:", model_data["customer_id"].nunique())
print("Repurchase rate:", model_data["repurchase"].mean())

Rows: 328787
Unique customers: 328787
Repurchase rate: 0.3791056215726291


In [43]:
assert model_data["customer_id"].is_unique
assert model_data["segment"].eq(2).all()
assert model_data["repurchase"].isin([0, 1]).all()

In [44]:
display(model_data.head())
display(model_data.isna().sum().to_frame("missing_values"))

,customer_id,segment,repurchase,recency,log_frequency,log_monetary,regular_fashion_news,club_member_active,product_group_entropy,both_channels,items_per_shopping_day,channel_2_item_share
2,00007e8d4e54114b5b2a9b51586325a8d0fa74ea23ef77...,2,0,141,0.693147,0.051981,0,1,0.693147,0,2.000000,0.000
6,0001076e215991bad544dd3e7312f78d9f576a1cc3ddc4...,2,1,101,1.098612,0.099830,0,1,1.011404,0,3.000000,0.000
8,0001420326c217a367048472e05395373c40a039f6fd45...,2,0,110,0.693147,0.087530,0,1,0.636514,0,3.000000,1.000
12,0001b0127d3e5ff8dadcfc6e5043682dba2070f2667081...,2,1,97,1.609438,0.149836,1,1,1.039721,1,2.000000,0.125
13,0001f8cef6b9702d54abf66fd89eb21014bf98567065a9...,2,1,101,1.386294,0.064231,0,1,1.004242,0,2.333333,0.000


,missing_values
customer_id,0
segment,0
repurchase,0
recency,0
log_frequency,0
log_monetary,0
regular_fashion_news,0
club_member_active,0
product_group_entropy,0
both_channels,0


## 9. Export Dataset

In [45]:
model_data.to_csv("/Volumes/USB/H&M Data/recent_occasional_logistic_regression_input.csv", index=False)